A) Carregar CSV com empreendimentos com lat/long preenchidas na etapa anterior

In [ ]:
# === CÉLULA A = Carregar CSV de válidos (com lat/long) ===

import pandas as pd

csv_validos = input("Informe o caminho do CSV de válidos (gerado na etapa anterior): ").strip()

try:
  df_validos = pd.read_csv(csv_validos, dtype=str, sep=';', encoding="utf-8")
  print(f"CSV carregado: {len(df_validos)} linhas")
except Exception as e:
  print("Erro ao ler CSV:", e)
  raise SystemExit

# Renomeação das colunas
df_validos = df_validos.rename(columns={
    "cod_operacao": "apf",
    "txt_nome_empreendimento": "empreendimento",
    "txt_endereco": "endereco",
    "txt_nome_municipio": "cidade",
    "txt_sigla_uf": "uf",
    "txt_modalidade": "recurso"
})

# sanitização
for c in ["lat", "long"]:
  df_validos[c] = pd.to_numeric(df_validos[c], errors="coerce")
df_validos = df_validos.dropna(subset=["lat", "long"]).reset_index(drop=True)
df_validos.head()

B) Utilitários e constantes

In [ ]:
# CÉLULA B - Utilitários e Constantes (pipeline em lote) ===

import geopandas as gpd
import networkx as nx
import osmnx as ox
from shapely.geometry import Point, LineString, MultiLineString
from shapely.ops import linemerge
from datetime import datetime
from pathlib import Path
import unicodedata
from collections import Counter

# --- Configurações OSMnx ---
ox.settings.use_cache = True
ox.settings.log_console = False

# --- Distâncias e cutoffs ---
RAIO_REDE_M = 2500 #raio para baixar a rede caminhável
DIST_C_D_E = 1000 #coletas: transporte (C), cotidianos obrigatórios (D), cotidianos complementares (E)
DIST_F_G = 1400 # coletas: eventuais obrigatórios (F), eventuais complementares (G)

CATEGORY_CUTOFFS = {
    "transporte": 1000,
    "cotidianos_obrigatorios": 1000,
    "cotidianos_complementares": 1000,
    "eventuais_obrigatorios": 1400,
    "eventuais_complementares": 1400,
}

TOL = 5 #tolerância para 1000m
TOL_EVT = 10 # tolerância para 1400m

# --- Tags OSM (C) para Transporte ---
POI_TAGS_TRANS = {
    "highway": ["bus_stop"],
    "amenity": ["bus_station"],
    "railway": ["station", "halt", "tram_stop", "subway_entrance"],
}

# --- Tags OSM (D) para Cotidianos Obrigatórios ---
POI_TAGS_COTIDIANOS_OBRIGATORIOS = {
    "leisure": ["playground", "recreation_ground", "park", "pitch", "sports_centre"],
    "amenity": ["marketplace", "childcare", "kindergarten", "school"],
    "shop": ["supermarket", "greengrocer", "convenience"],
}

# --- Tags OSM (E) para Cotidianos Complementares ---
POI_TAGS_COTIDIANOS_COMPLEMENTARES = {
    "shop": [
        "butcher", "bakery", "chemist", "hairdresser",
        "bicycle", "car_repair", "electronics", "electronics_repair",
        "repair", "household", "doityourself", "hardware", "lottery",
    ],
    "amenity": [
        "pharmacy", "restaurant", "atm", "gym" 
    ],
    "leisure": [
        "fitness_centre",
    ],
}

# --- Tags OSM (F) Eventuais Obrigatórios ---
POI_TAGS_EVENTUAIS_OBRIGATORIOS = {
    "amenity": ["school", "hospital", "pharmacy", "atm"],
    "healthcare": ["hospital", "urgent_care", "centre"],
    "leisure": ["pitch", "sports_centre", "stadium", "track", "recreation_ground"],
    "shop": ["chemist", "supermarket", "hypermarket", "lottery"],
}

# --- Tags OSM Eventuais Complementares ---
POI_TAGS_EVENTUAIS_COMPLEMENTARES = {
    "amenity": [
        "university", "college", "library", "police", "post_office",
        "restaurant", "bank", "clinic", "doctors", "hospital",
        "dentist", "social_facility", "language_school", "music_school",
    ],

    "healthcare": [
        "hospital", "clinic", "doctor", "centre", "dentist",
        "physiotherapist", "laboratory", "specialist", "yes",
    ],

    "shop": [
        "clothes", "shoes", "boutique", "fashion",
        "electronics", "household", "furniture", "computer", "appliance",
        "books", "stationery",
        "bicycle", "car_repair", "electronics_repair", "repair",
        "department_store", "variety_store",
    ],

    "office": ["lawyer", "accountant", "architect", "it", "therapist"],
    "government": ["social_services"],
    "social_facility": ["outreach"],
}

# --- Helpers gerais ---
def _to_wgs84(gdf):
  if gdf is None or gdf.empty:
    return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
  if gdf.crs is None:
    return gdf.set_crs(epsg=4326)
  return gdf.to_crs(epsg=4326)

def _ensure_osm_multiindex(gdf):
  if gdf is None or gdf.empty:
    return gdf
  if isinstance(gdf.index, pd.MultiIndex):
    names = list(gdf.index.names or [])
    if names == ["element", "id"]:
      gdf = gdf.rename_axis(index={"element": "element_type","id":"osmid"})
      names = list(gdf.index.names or [])
    if set(names) == {"element_type", "osmid"} and names != ["element_type", "osmid"]:
      gdf = gdf.reorder_levels(["element_type", "osmid"]).sort_index()
    return gdf
  cols = set(gdf.columns)
  if {"element_type", "osmid"}.issubset(cols):
    return gdf.set_index(["element_type", "osmid"])
  if {"element", "id"}.issubset(cols):
    return gdf.set_index(["element","id"]).rename_axis(index={"element":"element_type","id":"osmid"})
  return gdf

def _norm_name(x):
  if x is None or isinstance(x, float):
    return ""
  s = str(x).strip().lower()
  s = unicodedata.normalize("NFKD", s)
  s = "".join(ch for ch in s if not unicodedata.combining(ch))
  return " ".join(s.split())

def _topo_dedup_node_in_poly_same_name(gdf):
  if gdf is None or gdf.empty:
    return gdf
  is_pt = gdf.geometry.geom_type.isin(["Point", "MultiPoint"])
  is_area = gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
  g_pts = gdf[is_pt].copy()
  g_area = gdf[is_area].copy()
  if g_pts.empty or g_area.empty:
    return gdf
  g_pts["__name_n"] = g_pts.get("name", "").apply(_norm_name)
  g_area["__name_n"] = g_area.get("name", "").apply(_norm_name)
  g_pts_n = g_pts[g_pts["__name_n"] != ""].copy()
  g_area_n = g_area[g_area["__name_n"] != ""].copy()
  if g_pts_n.empty or g_area_n.empty:
    return gdf
  g_pts_3857 = g_pts_n.to_crs(3857)
  g_area_3857 = g_area_n.to_crs(3857)
  try:
    joined = gpd.sjoin(g_pts_3857, g_area_3857, how="left", predicate="within")
    ln, rn = "__name_n_left", "__name_n_right"
  except Exception:
    joined = gpd.sjoin(g_pts_3857, g_area_3857, how="left")
    ln = "__name_n"
    rn = "__name_n_right" if "__name_n_right" in joined.columns else "__name_n_1" if "__name_n_1" in joined.columns else "__name_n"
  if rn not in joined.columns:
    return gdf
  same_name = (joined[ln].notna() & joined[rn].notna() & (joined[ln] == joined[rn]))
  to_drop_pts_idx = joined[same_name].index.unique()
  g_pts_keep = g_pts.drop(index=to_drop_pts_idx, errors="ignore")
  keep_idx = list(g_pts_keep.index) + list(gdf[~(is_pt|is_area)].index) + list(g_area.index)
  return gdf.loc[keep_idx].copy()

def _make_geom_pt(gdf):
  """
  Garante uma coluna 'geom_pt' com um ponto representativo do POI, SEMPRE em EPSG:4326.
  - Se a geometria já for Point / MultiPoint: usa o próprio ponto
  - Caso contrário (Polygon, LineString etc.): usa representative_point() (ponto interno)  
  """
  if gdf is None or gdf.empty:
    g = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
    g["geom_pt"] = None
    return g
  
  # Garante que o GeoDataFrame esteja em WGS84
  gdf = gdf.to_crs(4326)

  geom = gdf.geometry
  geom_pt = geom.copy()

  # Identifica geometrias que nÃO são ponto
  mask_not_pt = ~geom.geom_type.isin(["Point", "MultiPoint"])

  if mask_not_pt.any():
    # Calcula centróides somente para as geometrias necessárias
    geom_pt.loc[mask_not_pt] = geom.loc[mask_not_pt].representative_point()
  
  # Cria coluna explícita de ponto representativo
  gdf = gdf.copy()
  gdf["geom_pt"] = geom_pt
  return gdf

def _features_from_point(lat, lon, tags, dist):
  try:
    g = ox.features.features_from_point((lat, lon), tags=tags, dist=dist)
  except Exception:
    g = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")
  g = _to_wgs84(g)
  g = _ensure_osm_multiindex(g)
  return g

# --- Helpers para persistir/recuperar rotas por APF (JSON) ---
import json
from glob import glob
import math
import numpy as np

def _to_serializable_routes(route_results):
  """
  Converte ROUTE_RESULTS em uma lista 100% serializável em JSON:
  - remove pt_wgs (guarda lat/lon nativos)
  - converte numpy.* para tipos Python (bool/int/foat)
  - normaliza coords_nodes/leg_coords para [[lat, lon], ...] com float nativo
  - garante node_path como lista de int nativos
  """
  def _to_float(x):
    try:
      if x is None or (isinstance(x, float) and math.isnan(x)):
        return None
      if isinstance(x, (np.floating,)):
        x = float(x)
      return float(x)
    except Exception:
      return None

  def _to_int(x):
    try:
      if x is None:
        return None
      if isinstance(x, (np.integer,)):
        x = int(x)
      return int(x)
    except Exception:
      return None

  def _to_bool(x):
    try:
      if x is None:
        return False
      return bool(x)
    except Exception:
      return False

  def _to_str(x):
    if x is None:
      return None
    try:
      s = str(x)
      return s
    except Exception:
      return None

  safe = []
  for r in route_results:
    # lat/lon a partir do pt_wgs
    pt = r.get("pt_wgs")
    lat = _to_float(getattr(pt, "y", None)) if pt is not None else _to_float(r.get("lat"))
    lon = _to_float(getattr(pt, "x", None)) if pt is not None else _to_float(r.get("lon"))

    # coords_nodes (de CRS métrico para WGS84)
    coords_nodes_proj = r.get("coords_nodes_proj") or []
    coords_nodes = []

    if coords_nodes_proj:
      gdf_nodes = gpd.GeoSeries(coords_nodes_proj, crs=3857).to_crs(4326)
      coords_nodes = [[float(pt.y), float(pt.x)] for pt in gdf_nodes]

    # perninha final (de CRES métrico para WGS84)
    leg_geom_proj = r.get("leg_geom_proj")
    leg_coords = []
    
    if leg_geom_proj is not None:
      gdf_leg = gpd.GeoSeries([leg_geom_proj], crs=3857).to_crs(4326)
      leg_coords = [[float(y), float(x)] for x, y in gdf_leg.iloc[0].coords]

    # node_path
    node_path = r.get("node_path")
    if isinstance(node_path, (list, tuple)):
      node_path = [_to_int(n) for n in node_path if _to_int(n) is not None]
    else:
      node_path = []

    #ids e atributos simples
    element_type = _to_str(r.get("element_type"))
    osmid = _to_int(r.get("osmid"))
    tipo_bruto = _to_str(r.get("tipo_bruto"))
    nome = _to_str(r.get("nome"))
    categoria = _to_str(r.get("categoria"))
    d_net_m = _to_float(r.get("d_net_m"))
    within = _to_bool(r.get("within_cutoff"))

    rec = {
        "categoria": categoria,
        "element_type": element_type,
        "osmid": osmid,
        "tipo_bruto": tipo_bruto,
        "nome": nome,
        "lat": lat,
        "lon": lon,
        "d_net_m": d_net_m,
        "within_cutoff": within,
        "node_path": node_path,
        "coords_nodes": coords_nodes,
        "leg_coords": leg_coords,
    }
    safe.append(rec)
  return safe

def save_routes_json(apf, route_results, folder: Path):
  folder.mkdir(parents=True, exist_ok=True)
  ts = datetime.now().strftime("%Y%m%d_%H%M%S")
  fname = folder / f"routes_{str(apf).strip()}_{ts}.json"
  data = _to_serializable_routes(route_results)

  # default para qualquer resquício de tipo não serializável (fallback)
  def _json_default(o):
    try:
      if isinstance(o, np.integer):
        return int(o)
      if isinstance (o, np.floating):
        return float(o)
      if isinstance(o, np.bool_):
        return bool(o)
      return str(o)
    except Exception:
      return str(o)

  with open(fname, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, default=_json_default)
  return fname

def load_latest_routes_json(apf, base_folder="."):
  """
  Carrega o JSON de rotas MAIS RECENTE do APF.
  retorna lista de dicts para a CÉLULA 4 (com lat/lon; sem pt_wgs).
  """
  apf_str = str(apf).strip()
  pattern = str(Path(base_folder) / apf_str / f"routes_{apf_str}_*.json")
  files = sorted(glob(pattern))
  if not files:
    return None, None #(None, None) => não há arquivo para esse APF
  latest = files[-1]
  with open(latest, "r", encoding="utf-8") as f:
    data = json.load(f)
  return data, latest

# --- Coletas (C/D/E/F/G em funções) ---
def coleta_transporte(lat,lon):
  g = _features_from_point(lat, lon, POI_TAGS_TRANS, DIST_C_D_E)
  g["uso"] = "transporte"
  if g.index.has_duplicates:
    g = g[~g.index.duplicated(keep="first")].copy()
  g = _make_geom_pt(g)
  return g

def coleta_cotidianos_obrigatorios(lat,lon):
  g = _features_from_point(lat, lon, POI_TAGS_COTIDIANOS_OBRIGATORIOS, DIST_C_D_E)
  if g.empty:
    g = _make_geom_pt(g); g["uso"]=None; return g
  g["uso"] = "cotidianos_obrigatorios"
  if g.index.has_duplicates:
    g = g[~g.index.duplicated(keep="first")].copy()
  g = _topo_dedup_node_in_poly_same_name(g)
  g = _make_geom_pt(g)
  return g

def coleta_cotidianos_complementares(lat,lon):
  g = _features_from_point(lat, lon, POI_TAGS_COTIDIANOS_COMPLEMENTARES, DIST_C_D_E)
  if g.empty:
    g = _make_geom_pt(g); g["uso"]=None; return g
  g["uso"] = "cotidianos_complementares"
  if g.index.has_duplicates:
    g = g[~g.index.duplicated(keep="first")].copy()
  g = _topo_dedup_node_in_poly_same_name(g)
  g = _make_geom_pt(g)
  return g

def coleta_eventuais_obrigatorios(lat, lon):
  g = _features_from_point(lat, lon, POI_TAGS_EVENTUAIS_OBRIGATORIOS, DIST_F_G)
  if g.empty:
    g = _make_geom_pt(g); g["uso"]=None; return g
  g["uso"] = "eventuais_obrigatorios"
  if g.index.has_duplicates:
    g = g[~g.index.duplicated(keep="first")].copy()
  g = _topo_dedup_node_in_poly_same_name(g)
  g = _make_geom_pt(g)
  return g

def coleta_eventuais_complementares(lat, lon):
  g = _features_from_point(lat, lon, POI_TAGS_EVENTUAIS_COMPLEMENTARES, DIST_F_G)
  if g.empty:
    g = _make_geom_pt(g); g["uso"]=None; return g
  g["uso"] = "eventuais_complementares"
  if g.index.has_duplicates:
    g = g[~g.index.duplicated(keep="first")].copy()
  g = _topo_dedup_node_in_poly_same_name(g)
  g = _make_geom_pt(g)
  return g

# --- Rede ---
def build_network(lat, lon, raio_m=RAIO_REDE_M):
    # 1) Grafo em WGS84
    G_raw = ox.graph_from_point((lat, lon), dist=raio_m, network_type="walk")
    G_raw = ox.distance.add_edge_lengths(G_raw)

    # 2) Reprojetar grafo para CRS métrico
    G_proj = ox.project_graph(G_raw, to_crs="EPSG:3857")

    # 3) Converter ponto (lon, lat) para o mesmo CRS métrico do grafo
    pt_proj = (
        gpd.GeoSeries([Point(lon, lat)], crs="EPSG:4326")
        .to_crs(G_proj.graph["crs"])
        .iloc[0]
    )

    # 4) nó central a partir do empreendimento
    center_node = ox.distance.nearest_nodes(G_proj, pt_proj.x, pt_proj.y)

    # 5) Grafo não direcionado
    Gu = G_proj.to_undirected()

    # 6) Distância a partir do centro (em METROS)
    dist_map = nx.single_source_dijkstra_path_length(Gu, center_node, weight="length")

    # 7) Apenas dados métricos
    edges_proj = ox.graph_to_gdfs(G_proj, nodes=False)

    # 8) Retorna G_projetado
    return dict(
        G=G_proj,
        Gu=Gu,
        center_node=center_node,
        dist_map=dist_map,
        edges_proj=edges_proj
    )

# --- Helpers de rota ---
def _ensure_linestring_containing_point(geom, p):
  if isinstance(geom, LineString):
    return geom
  if isinstance(geom, MultiLineString):
    parts = list(geom.geoms)
  elif hasattr(geom, "geoms"):
    parts = [g for g in geom.geoms if isinstance(g, (LineString, MultiLineString))]
  if not parts:
    return None
  merged = linemerge(parts)
  if isinstance(merged, LineString):
    return merged
  if isinstance(merged, MultiLineString):
    return min(list(merged.geoms), key=lambda g: g.distance(p))
  return min(parts, key=lambda g: g.distance(p))

def _cut_linestring_at_distance(line, d):
  if d <= 0.0:
    return LineString([]), LineString(line.coords)
  L = float(line.length)
  if d >= L:
    return LineString(line.coords), LineString([])
  coords = list(line.coords)
  acc = 0.0
  seg_coords = [coords[0]]
  for i in range(1, len(coords)):
    p0 = Point(coords[i-1]); p1 = Point(coords[i])
    seg_len = p0.distance(p1)
    if acc + seg_len ==d:
      seg_coords.append(coords[i])
      return LineString(seg_coords), LineString(coords[i:])
    if acc + seg_len > d:
      ratio = (d - acc) / seg_len
      ip = (p0.x + ratio*(p1.x - p0.x), p0.y + ratio*(p1.y - p0.y))
      seg_coords.append(ip)
      return LineString(seg_coords), LineString([ip] + coords[i:])
    seg_coords.append(coords[i]); acc += seg_len
  return LineString(line.coords), LineString([])

def _segment_to_point(line_proj, to_u, pt_proj):
  line_ls = _ensure_linestring_containing_point(line_proj, pt_proj)
  if line_ls is None or line_ls.is_empty:
    return None
  s = max(0.0, min(line_ls.length, line_ls.project(pt_proj)))
  part_u, part_v = _cut_linestring_at_distance(line_ls, s)
  return part_u if to_u else part_v

def _nearest_edge_by_sindex(pt_proj, egdf_proj):
  for radius in (10, 25, 50, 100, 150):
    buf = pt_proj.buffer(radius)
    try:
      idx = list(egdf_proj.sindex.query(buf, predicate="intersects"))
    except Exception:
      idx = list(egdf_proj.sindex.query(buf))
    if idx:
      subset = egdf_proj.iloc[idx].copy()
      subset["_dist"] = subset.geometry.distance(pt_proj)
      row = subset.sort_values("_dist").iloc[0]

      # --- ROBUSTEZ: extrai u/v de colunas OU do índice (MultiIndex) ---
      u = None; v = None
      # 1) tenta colunas
      try:
          if "u" in row.index:
            u = row["u"]
          if "v" in row.index:
            v = row["v"]
      except Exception:
          pass
      # 2) se não achou, tenta pelo índice (name)
      if (u is None or v is None):
        name = row.name
        if isinstance(name, tuple):
          if len(name) >=2:
            u = name[0] if u is None else u
            v = name[1] if v is None else v
      return u, v, row.geometry
  return None, None, None

def safe_str(x):
  if x is None: return ""
  if isinstance(x, float) and pd.isna(x): return ""
  try: return str(x)
  except Exception: return ""

def osm_tipo_bruto(rec):
  for k in ["highway", "amenity", "railway", "public_transport", "leisure", "shop", "healthcare", "office", "government", "social_facility"]:
    val = rec.get(k, None)
    if isinstance(val, str) and val.strip():
      return f"{k}={val}"
  for k in rec.index:
    if k in ["geometry", "geom_pt", "name", "uso"]:
      continue
    v = rec.get(k, None)
    if isinstance(v, str) and v.strip():
      return f"{k}={v}"
  return "tipo=desconhecido"

# --- Roteamento unificado por categoria ---
def calcular_rotas_categoria(gdf_entrada, categoria, infra):
  """
  gdf_entrada: GeoDataFrame com geom_pt
  categoria: str (ex.: 'cotidianos_obrigatorios')
  infra: dict retornado por build_network
  Retorna: lista de dicts (para acumular em ROUTE_RESULTS)
  """
  if gdf_entrada is None or gdf_entrada.empty:
    return []

  cutoff = CATEGORY_CUTOFFS.get(categoria, 1000)
  tol = TOL_EVT if cutoff >= 1400 else TOL

  edges_proj = infra["edges_proj"]
  dist_map = infra["dist_map"]
  Gu_center = infra["Gu"]
  center_node = infra["center_node"]
  
  # Grafo projetado (CRS métrico)
  G_proj = infra["G"]

  #Nós do grafo em CRS métrico (EPSG:3857)
  nodes_proj = ox.graph_to_gdfs(G_proj, edges=False)

  resultados = []
  for idx, r in gdf_entrada.iterrows():
    pt = r.get("geom_pt", None)
    if pt is None:
      continue
    if getattr(pt, "geom_type", "") == "MultiPoint":
      try: pt = list(pt.geoms)[0]
      except Exception:
        continue
    if getattr(pt, "geom_type", "") != "Point":
      continue

    # Ponto do POI em CRS métrico
    pt_proj = gpd.GeoSeries([pt], crs=4326).to_crs(3857).iloc[0]

    # Aresta mais próxima
    u, v, line_proj = _nearest_edge_by_sindex(pt_proj, edges_proj)
    if u is None or v is None or line_proj is None or line_proj.is_empty:
      continue

    du = dist_map.get(u, None)
    dv = dist_map.get(v, None)
    if du is None and dv is None:
      continue

    # Projeção do ponto na aresta
    L = float(line_proj.length)
    s = float(line_proj.project(pt_proj))
    s = max(0, min(L, s))

    if du is not None and dv is not None:
      cost_u = du + s
      cost_v = dv + (L - s)
      d_net = min(cost_u, cost_v)
      target = u if cost_u <= cost_v else v
      target_is_u = (target == u)
    elif du is not None:
      d_net = du + s
      target = u
      target_is_u = True
    else:
      d_net = dv + (L -s)
      target = v
      target_is_u = False

    # Caminho por nós (em CRS métrico)
    try:
      node_path = nx.shortest_path(Gu_center, center_node, target, weight="length")
    except Exception:
      node_path = None

    # coords por nós (em CRS métrico - EPSG:3857)
    coords_nodes_proj, ok_nodes = [], True
    if node_path:
      for n in node_path:
        try:
          coords_nodes_proj.append(nodes_proj.loc[n].geometry)
        except Exception:
          ok_nodes = False
          break

    # Perninha final (independente dos nós)
    leg_geom_proj = None
    if line_proj is not None and not line_proj.is_empty:
      seg_final_proj = _segment_to_point(line_proj, target_is_u, pt_proj)
      if seg_final_proj is not None and not seg_final_proj.is_empty:
        leg_geom_proj = seg_final_proj

    has_route_nodes = ok_nodes and len(coords_nodes_proj) >= 2
    has_leg = leg_geom_proj is not None

    if has_route_nodes or has_leg:
      #Extrai id OSM do índice
      element_type, osmid = (None, None)
      if isinstance(idx, tuple) and len(idx) == 2:
        element_type, osmid = idx

      resultados.append({
        "categoria": categoria,
        "element_type": element_type,
        "osmid": osmid,
        "tipo_bruto": osm_tipo_bruto(r),
        "nome": safe_str(r.get("name","")),
        "pt_wgs": pt, #mantido deliberadamente em EPSG:4326
        "d_net_m": float(d_net),
        "within_cutoff": (d_net <= (cutoff + tol)),
        "node_path": node_path,
        "coords_nodes_proj": coords_nodes_proj if has_route_nodes else [],
        "leg_geom_proj": leg_geom_proj if has_leg else None,
      })
  return resultados

# --- Exportação (agregado + detalhado) ---
def tipo_to_pair(tipo):
  if not isinstance(tipo, str) or "=" not in tipo:
    return None
  k, v = tipo.split("=", 1)
  return f"{k.strip()}:{v.strip()}"

def counts_por_categoria(df_ok, categoria):
  dfg = df_ok[df_ok["categoria"] == categoria]
  cnt = Counter()
  for tipo in dfg["tipo_bruto"].dropna().astype(str):
    pair = tipo_to_pair(tipo)
    if pair:
      cnt[pair] +=1
  if not cnt:
    return ""
  return " | ".join(f"{k}={v}" for k, v in sorted(cnt.items(), key=lambda x: (-x[1], x[0])))

TAG_COLS = ["amenity", "shop", "healthcare", "leisure", "office", "government", "social_facility"]
CAT_TO_GDF_NAME = {
    "transporte": "gdf_transp",
    "cotidianos_obrigatorios": "gdf_cotidianos_obrigatorios",
    "cotidianos_complementares": "gdf_cotidianos_complementares",
    "eventuais_obrigatorios": "gdf_eventuais_obrigatorios",
    "eventuais_complementares": "gdf_eventuais_complementares",
}

def exportar_agregado_e_detalhe(meta, route_results, gdfs_by_cat, base_folder_path: Path):
  """
  meta: dict com campos do empreendimento (apf, empreendimento, endereco, cidade, uf, recurso, lat, long)
  route_results: lista de dicts (ROUTE_RESULTS)
  gdfs_by_cat: dict categoria -> gdf (para enriquecer tags se possível)
  base_folder_path: Path da pasta do APF
  Retorna: (Path arquivo_agregado, Path arquivo_detalhe, dict registro_agregado)
  """
  df_routes = pd.DataFrame(route_results)
  #agregado só com dentro do cutoff
  df_ok = df_routes[df_routes["within_cutoff"] ==True].copy()

  registro = dict(meta)
  for cat in ["transporte","cotidianos_obrigatorios","cotidianos_complementares","eventuais_obrigatorios","eventuais_complementares"]:
    registro[f"{cat}_contagens"] = counts_por_categoria(df_ok, cat)

  df_agregado = pd.DataFrame([registro])

  #detalhado: dentro e fora
  def coords_from_pt(pt):
    try: return float(pt.y), float(pt.x)
    except Exception: return None, None

  det_rows = []
  for _, r in df_routes.iterrows():
    cat = r.get("categoria")
    lat_poi, lon_poi = coords_from_pt(r.get("pt_wgs"))
    et, oid = r.get("element_type"), r.get("osmid")
    osm_url = f"https://www.openstreetmap.org/{et}/{oid}" if et and (oid is not None) else None
    det = {
        "apf": meta["apf"],
        "categoria": cat,
        "element_type": et,
        "osmid": oid,
        "tipo_bruto": r.get("tipo_bruto"),
        "nome": r.get("nome"),
        "lat": lat_poi,
        "long": lon_poi,
        "d_net_m": r.get("d_net_m"),
        "within_cutoff": bool(r.get("within_cutoff")),
        "distancia_cutoff_m": CATEGORY_CUTOFFS.get(cat, None),
        "osm_url": osm_url,
    }
    # enriquecer com tags originais (se gdf disponível)
    gdf_ref = gdfs_by_cat.get(cat)
    if gdf_ref is not None and et is not None and oid is not None:
      try:
        if hasattr(gdf_ref, "index") and gdf_ref.index.nlevels == 2 and (et, oid) in gdf_ref.index:
          row_ref = gdf_ref.loc[(et, oid)]
          for col in TAG_COLS:
            if col in gdf_ref.columns:
              det[col] = row_ref.get(col, None)
      except Exception:
        pass
    det_rows.append(det)

  df_detalhe = pd.DataFrame(det_rows)

  # nomes e gravação
  ts = datetime.now().strftime("%Y%m%d_%H%M%S")
  apf_str = str(meta["apf"]).strip()
  base_folder_path.mkdir(parents=True, exist_ok=True)
  arq_agregado = base_folder_path / f"resumo_agregado_pois_{apf_str}_{ts}.csv"
  arq_detalhe = base_folder_path / f"pois_detalhe_{apf_str}_{ts}.csv"

  df_agregado.to_csv(arq_agregado, index=False, encoding="utf-8", sep=";", decimal=",")
  df_detalhe.to_csv(arq_detalhe, index=False, encoding="utf-8", sep=";", decimal=",")

  return arq_agregado, arq_detalhe, registro

CÉLULA C - Processar TODOS os empreendimentos

Itera sobre todas as linhas de df_validos (carregado na CÉLULA A) e, para cada APF:

Coleta POIs (transporte C, cotidianos obrigatórios D, cotidianos complementares E, eventuais obrigatórios F, eventuais complementares G);
Exclui por OSM ID entre categorias onde aplicável (F exclui D/E; G exclui D/E/F);
Constrói a rede;
Calcula as rotas para todas as categorias com dados;
Exporta dois arquivos na pasta <APF>/;
Acumula um registro agregado para o CSV geral com uma linha por APF.

In [ ]:
# === CÉLULA C - Processar TODOS os empreendimentos de df_validos ===
# Requer: df_validos carregado na CÉLULA A e CÉLULA B acima

from pathlib import Path

assert 'df_validos' in globals() and len(df_validos) > 0, "df_validos não carregado."

# Lista de registros agregados *uma linha por APF) para o arquivo geral
registros_geral = []

for i, linha in df_validos.iterrows():
  # Metadados do Empreendimento
  apf = linha["apf"]
  emp = linha["empreendimento"]
  lat = float(linha["lat"])
  lon = float(linha["long"])
  meta = {
      "apf": apf,
      "empreendimento": emp,
      "endereco": linha.get("endereco"),
      "cidade": linha.get("cidade"),
      "uf": linha.get("uf"),
      "recurso": linha.get("recurso"),
      "lat": lat,
      "long": lon,
  }

  print(f"\n=== [{i+1}/{len(df_validos)}] APF {apf} - {emp} ({meta['cidade']}/{meta['uf']}) ===")

  # 1) Coletas (C, D, E, F, G)
  gdf_transp = coleta_transporte(lat, lon)
  gdf_D = coleta_cotidianos_obrigatorios(lat, lon) # D
  gdf_E = coleta_cotidianos_complementares(lat, lon) # E
  gdf_F = coleta_eventuais_obrigatorios(lat, lon)
  gdf_G = coleta_eventuais_complementares(lat, lon)
  print(f"Coletas iniciais: T={len(gdf_transp)} | D={len(gdf_D)} | E={len(gdf_E)} | F={len(gdf_F)} | G={len(gdf_G)}")

  # 2) Rede
  infra = build_network(lat, lon, RAIO_REDE_M)
  print(f"Rede: nodes={len(infra['Gu'])} | edges={infra['Gu'].number_of_edges()}")

  # 3) Rotas 'brutas' (com duplicidades)
  ROUTE_RESULTS = [] # Zera a cada APF

  def add_cat(cat_name, gdf_cat):
    before = len(ROUTE_RESULTS)
    ROUTE_RESULTS.extend(calcular_rotas_categoria(gdf_cat, cat_name, infra))
    added = len(ROUTE_RESULTS) - before
    print(f"Rotas {cat_name}: +{added}")

  add_cat("transporte", gdf_transp)
  add_cat("cotidianos_obrigatorios", gdf_D)
  add_cat("cotidianos_complementares", gdf_E)
  add_cat("eventuais_obrigatorios", gdf_F)
  add_cat("eventuais_complementares", gdf_G)

  print(f"Total ROUTE_RESULTS: {len(ROUTE_RESULTS)}")

  # 4) Exclusão de duplicidades entre D/E/F/G

  # Ordem de prioridade (quanto menor o índice, maior a prioridade)
  PRIORIDADE_CATEGORIA = {
    "cotidianos_obrigatorios": 1,   # D
    "cotidianos_complementares": 2, # E
    "eventuais_obrigatorios": 3,    # F
    "eventuais_complementares": 4,  # G
  }

  from collections import defaultdict

  # Agrupa todas as rotas por POI (element_type + osmid)
  por_poi = defaultdict(list)

  for r in ROUTE_RESULTS:
    et = r.get("element_type")
    oid = r.get("osmid")

    # Transporte ou POIS sem ID ficam fora da lógica de conflito
    if r.get("categoria") == "transporte" or et is None or oid is None:
      por_poi[("transporte", id(r))].append(r)
    else:
      por_poi[(et, oid)].append(r)
    
  ROUTE_RESULTS_RESOLVIDO = []

  for key, registros in por_poi.items():

    # caso transporte ou POI sem OSMID -> mantém tudo como está
    if key[0] == "transporte":
      ROUTE_RESULTS_RESOLVIDO.extend(registros)
      continue

    # Ordena por prioridade de categoria (D>E>F>G)
    registros_ordenados = sorted(
      registros,
      key=lambda r: PRIORIDADE_CATEGORIA.get(r["categoria"], 99)
    )

    # Procura a primeira categoria dentro do cutoff
    escolhido = None
    for r in registros_ordenados:
      if r.get("within_cutoff"):
        escolhido = r
        break
    
    if escolhido is not None:
      # Mantém apenas a categoria válida
      ROUTE_RESULTS_RESOLVIDO.append(escolhido)
    else:
      # Nenhum ficou dentro do cutoff
      # Mantém apenas a de maior prioridade (para visualização em mapa)
      ROUTE_RESULTS_RESOLVIDO.append(registros_ordenados[0])

  # Substitui o conjunto original
  ROUTE_RESULTS = ROUTE_RESULTS_RESOLVIDO

  print(f"Após exclusão de duplicidades: {len(ROUTE_RESULTS)} rotas")


  # 5) Exportações
  gdfs_by_cat = {
      "transporte": gdf_transp,
      "cotidianos_obrigatorios": gdf_D,
      "cotidianos_complementares": gdf_E,
      "eventuais_obrigatorios": gdf_F,
      "eventuais_complementares": gdf_G,
  }
  pasta_apf = Path(str(apf).strip())

  # CSV agregado + detalhado
  arq_agregado, arq_detalhe, registro_agregado = exportar_agregado_e_detalhe(meta, ROUTE_RESULTS, gdfs_by_cat, pasta_apf)

  #JSON de rotas (para abrir no mapa depois, sem recalcular)
  routes_json = save_routes_json(apf, ROUTE_RESULTS, pasta_apf)

  registros_geral.append(registro_agregado)

  print(f"Gerado:\n - {arq_agregado}\n - {arq_detalhe}")

# 6) CSV geral (uma linha por APF) na raiz
if registros_geral:
  ts = datetime.now().strftime("%Y%m%d_%H%M%S")
  arq_geral = Path(f"resumo_agregado_pois_GERAL_{ts}.csv")
  pd.DataFrame(registros_geral).to_csv(arq_geral, index=False, encoding="utf-8", sep=";")
  print(f"\nArquivo GERAL gerado: {arq_geral.resolve()}")
else:
  print("\nNenhuma linha gerada no resumo geral.")

Célula D - escolha de empreendimento para visualização em mapa

In [ ]:
# === CÉLULA D - Selecionar APF e carregar rotas salvas (para o mapa) ===

assert 'df_validos' in globals(), "df_validos não encontrado - execute a CÉLULA A."

print("Empreendimentos disponíveis:\n")
print(df_validos[["apf", "empreendimento", "cidade", "uf"]])

idx_in = input("\nDigite o ÍNDICE (linha) do empreendimento para visualizar o mapa: ").strip()
try:
  idx = int(idx_in)
except Exception:
  raise SystemExit("Índice inválido (não numérico).")

if idx < 0 or idx >= len(df_validos):
  raise SystemExit(f"Índice inválido (fora do intervalo 0..{len(df_validos)-1}).")

linha = df_validos.loc[idx]
apf = linha["apf"]
emp = linha["empreendimento"]
lat = float(linha["lat"])
lon = float(linha["long"])

#Carrega o JSON de rotas mais recente para este APF
ROUTE_RESULTS, routes_file = load_latest_routes_json(apf)
if ROUTE_RESULTS is None:
  raise SystemExit(f"Não foi encontrado arquivo de rotas JSON para o APF {apf}. "
                    "Execute o processamento em lote antes.")

print(f"\nRotas carregadas do arquivo: {routes_file}")
print(f"APF {apf} - {emp}")
print(f"Total de linhas em ROUTE_RESULTS (para o mapa): {len(ROUTE_RESULTS)}")

Célula E: exibição do mapa com Folium

In [ ]:
# === CÉLULA E - Mapa final (consome ROUTE_RESULTS e desenha) ===

import folium

assert 'ROUTE_RESULTS' in globals() and len(ROUTE_RESULTS) > 0, "Sem resultados acumulados. Execute as células de cálculo antes."

# Paleta por categoria
CATEGORY_COLORS = {
    "transporte": {"within": "black", "beyond": "lightgray"},
    "cotidianos_obrigatorios": {"within": "blue", "beyond": "lightblue"},
    "cotidianos_complementares": {"within": "green", "beyond": "lightgreen"},
    "eventuais_obrigatorios": {"within": "darkred", "beyond": "lightred"},
    "eventuais_complementares": {"within": "orange", "beyond": "beige"},
}

CATEGORY_ICONS = {
    "transporte": "taxi",
    "cotidianos_obrigatorios": "star",
    "cotidianos_complementares": "star-half",
    "eventuais_obrigatorios": "plus",
    "eventuais_complementares": "minus",
}

# Formata o cutoff para a legenda: "1,0 km", "1,4 km"
def fmt_cutoff(m):
  km = m/1000
  return f"{km:.1f} km".replace(".", ",")

# Mapa base
m_final = folium.Map(
    location=[float(lat), float(lon)],
    tiles="cartodbpositron", #"OpenStreetMap",
    zoom_start=15,
    control_scale=True
)

# Marcador do empreendimento
titulo_emp = f"{apf} - {emp}" if "apf" in globals() and "emp" in globals() else "Empreeendimento"
folium.Marker(
    location=[float(lat), float(lon)],
    tooltip=titulo_emp,
    icon=folium.Icon(color="red", icon="home")
).add_to(m_final)

# Camadas por categoria
layers = {}

for res in ROUTE_RESULTS:
  categoria = res["categoria"]
  colors = CATEGORY_COLORS.get(categoria, {"within": "blue", "beyond": "gray"})
  color_line = "#444444"

  cutoff_m = CATEGORY_CUTOFFS.get(categoria, 1000)
  cutoff_label = fmt_cutoff(cutoff_m)

  # cria camadas da categoria com legenda específica
  if categoria not in layers:
    layers[categoria] = {
        "within": folium.FeatureGroup(name=f"{categoria} - ≤ {cutoff_label}", show=True),
        "beyond": folium.FeatureGroup(name=f"{categoria} - > {cutoff_label}", show=True),
    }

  within_key = "within" if res["within_cutoff"] else "beyond"
  pin_color = colors[within_key]

  # Rota por nós
  coords_nodes = res["coords_nodes"]
  if coords_nodes and len(coords_nodes) >= 2:
    folium.PolyLine(
        locations=coords_nodes, color=color_line, weight=3, opacity=0.9
    ).add_to(layers[categoria][within_key])

  # Perninha final
  if res["leg_coords"] and len(res["leg_coords"]) >= 2:
    folium.PolyLine(
        locations=res["leg_coords"], color=color_line, weight=3, opacity=0.9
    ).add_to(layers[categoria][within_key])

  # Pino do POI (com link para o OSM)
  lat_pt = res.get("lat")
  lon_pt = res.get("lon")
  tipo = res["tipo_bruto"]
  nome = res["nome"]
  d_txt = f" - {int(round(res['d_net_m']))} m"

  # IDs OSM
  et = res.get("element_type")
  oid = res.get("osmid")
  osm_url = f"https://www.openstreetmap.org/{et}/{oid}" if et and oid is not None else None

  # HTML do popup
  popup_html = f"<b>{tipo}</b>"
  if nome:
    popup_html += f"<br><b>name:</b>{nome}"
  if osm_url:
    popup_html += f"<br><a href='{osm_url}' target='_blank'>Abrir no OpenStreetMap</a>"

  folium.Marker(
      location=(lat_pt, lon_pt),
      tooltip=f"{tipo}{d_txt}",
      popup=folium.Popup(popup_html, max_width=302),
      icon=folium.Icon(color=pin_color, icon=CATEGORY_ICONS.get(categoria, "info-sign"), prefix="fa")
  ).add_to(layers[categoria][within_key])

# Adicionar camadas e controle
for cat, group in layers.items():
  group["within"].add_to(m_final)
  group["beyond"].add_to(m_final)

folium.LayerControl(position="topright", collapsed=False).add_to(m_final)

# Exibir mapa
m_final

In [ ]:
# CÉLULA F - Salvar mapa HTML na pasta do APF

# Garante que o mapa e o arquivo de rotas existam
assert 'm_final' in globals(), "Mapa não encontrado (execute a Célula E)."
assert 'routes_file' in globals() and routes_file is not None, "Arquivo de rotas não encontrado."

# Pasta do APF (a mesma onde está o JSON de rotas)
pasta_apf = Path(routes_file).parent

# Timestamp para o nome do arquivo
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# Nome do arquivo HTML
nome_arquivo = f"mapa_pois_{str(apf).strip()}_{ts}.html"

# Caminho completo
caminho_html = pasta_apf / nome_arquivo
#Salvando o mapa
m_final.save(caminho_html)

print(f"Mapa salvo em:\n{caminho_html.resolve()}")